# Advanced 01 — Measure-Theoretic Foundations of Probability

Practice notebook for the **"Measure-Theoretic Foundations of Probability"** section of the stats course PDF.

## What you will learn

This notebook recaps the theory from the PDF section, then **verifies it with Python**.
Each part ends with a **"Your turn"** prompt; the full **Problem Set** at the end has
**click-to-reveal solutions**.

## How to use

1. **Read** each markdown cell, then **run** the code beneath it (`Shift+Enter`).
2. **Change parameters** and re-run — statistics is about *relationships*, not memorized formulas.
3. LaTeX uses \( \) for inline math and \[ \] / $$ $$ for display math (KaTeX-friendly).

Let's begin.


---
## Setup — run this first

NumPy for numerics, SciPy for exact distributions, Matplotlib for plots.


In [ ]:
# If anything is missing, uncomment and run once:
# !pip install numpy scipy matplotlib

import numpy as np
import matplotlib.pyplot as plt
import scipy
from scipy import stats

np.random.seed(42)
rng = np.random.default_rng(42)
plt.rcParams["figure.figsize"] = (9, 5)
plt.rcParams["axes.grid"] = True

print("Ready. NumPy", np.__version__, "| SciPy", scipy.__version__)


---
## Part 1 — Sigma-algebras and probability spaces

A **sigma-algebra** on a non-empty set \(\Omega\) is a collection
\(\mathcal{F}\subseteq 2^{\Omega}\) such that:

1. \(\Omega\in\mathcal{F}\).
2. Closed under complements: \(A\in\mathcal{F}\Rightarrow A^{c}\in\mathcal{F}\).
3. Closed under countable unions: \(A_1,A_2,\dots\in\mathcal{F}\Rightarrow\bigcup_{n}A_n\in\mathcal{F}\).

A **measure** \(\mu:\mathcal{F}\to[0,\infty]\) satisfies
\(\mu(\varnothing)=0\) and countable additivity on disjoint sets. If
\(\mu(\Omega)=1\) it is a **probability measure** and the triple
\((\Omega,\mathcal{F},P)\) is a **probability space**.

$$
(\Omega,\mathcal{F},P),\qquad
P(\Omega)=1,\qquad
P\Big(\bigcup_{n} A_n\Big)=\sum_{n}P(A_n)\ \text{for disjoint } A_n.
$$

On a *finite* sample space the countable conditions collapse to finite ones, so
we can build \(\mathcal{F}\) explicitly and check the axioms by enumeration.
Below we take \(\Omega=\{1,2,3,4\}\) (think: faces of a loaded die) and the
sigma-algebra generated by the partition \(\{\{1,2\},\{3,4\}\}\).


In [ ]:
# Finite probability space: Omega = {1,2,3,4}, F = sigma({1,2},{3,4})
import itertools
from itertools import chain, combinations

Omega = frozenset({1, 2, 3, 4})

def powerset(it):
    s = list(it)
    return list(chain.from_iterable(
        combinations(s, r) for r in range(len(s) + 1)
    ))

# Generator of a sigma-algebra from a partition: all unions of partition blocks.
partition = [frozenset({1, 2}), frozenset({3, 4})]

def sigma_from_partition(blocks):
    '''Smallest sigma-algebra containing the given blocks: all unions of blocks.'''
    out = []
    for choice in powerset(range(len(blocks))):
        U = frozenset().union(*(blocks[i] for i in choice)) if choice else frozenset()
        out.append(U)
    return out

F = sigma_from_partition(partition)
F_sorted = sorted(F, key=lambda s: (len(s), sorted(s)))
print("Omega =", sorted(Omega))
print("F (sigma-algebra) has", len(F_sorted), "events:")
for A in F_sorted:
    print("  ", sorted(A))

# Check the three sigma-algebra axioms explicitly.
assert Omega in F, "Omega must be in F"
assert all((Omega - A) in F for A in F), "F must be closed under complements"
# Finite sample space => countable union reduces to finite union.
union_closed = True
for A, B in itertools.product(F, F):
    if (A | B) not in F:
        union_closed = False
        break
assert union_closed, "F must be closed under finite unions"
print("Axioms OK: Omega in F, closed under complements, closed under finite unions.")


---
## Part 2 — Verifying the probability-measure axioms

A probability measure \(P:\mathcal{F}\to[0,1]\) must satisfy:

- **Non-negativity & normalization:** \(P(A)\geq 0\) for all \(A\in\mathcal{F}\) and \(P(\Omega)=1\).
- **Countable additivity (finite here):** for disjoint \(A_1,\dots,A_n\in\mathcal{F}\),
  \(P\!\left(\bigcup_k A_k\right)=\sum_k P(A_k)\).

We assign masses \(p_\omega\geq 0\) with \(\sum_\omega p_\omega=1\) and define
\(P(A)=\sum_{\omega\in A}p_\omega\). We then verify all three axioms by brute
force over every disjoint subfamily of \(\mathcal{F}\).


In [ ]:
# Define a probability measure P on (Omega, F) via atomic masses.
masses = {1: 0.10, 2: 0.20, 3: 0.30, 4: 0.40}   # sums to 1
assert abs(sum(masses.values()) - 1.0) < 1e-12

def P(A):
    return float(sum(masses[w] for w in A))

# Normalization + non-negativity
assert all(P(A) >= 0 for A in F), "P must be non-negative"
assert abs(P(Omega) - 1.0) < 1e-12, "P(Omega) must equal 1"

# Countable additivity on finite partitions: check EVERY disjoint subfamily.
def disjoint_subfamilies(events):
    events = list(events)
    for r in range(2, len(events) + 1):
        for combo in combinations(events, r):
            if all(a.isdisjoint(b) for a, b in combinations(combo, 2)):
                yield combo

additivity_ok = True
for fam in disjoint_subfamilies(F):
    union = frozenset().union(*fam)
    if abs(P(union) - sum(P(A) for A in fam)) > 1e-12:
        additivity_ok = False
        break
assert additivity_ok, "Countable (finite) additivity must hold for disjoint families"

print("Event      P(A)")
for A in F_sorted:
    print(f"  {str(sorted(A)):>14}  {P(A):.2f}")
print()
print("Axioms OK: non-negativity, normalization P(Omega)=1, finite additivity.")


---
## Part 3 — Random variables as measurable maps

A function \(X:(\Omega,\mathcal{F})\to(\mathbb{R},\mathcal{B})\) is a **random
variable** if it is measurable, i.e. the preimage of every Borel set lies in
\(\mathcal{F}\):

$$
X\ \text{measurable}\quad\Longleftrightarrow\quad
\{\omega:X(\omega)\leq x\}\in\mathcal{F}\ \ \forall x\in\mathbb{R}.
$$

Equivalently (for a finite \(\Omega\) with \(\mathcal{F}\) generated by a
partition), \(X\) is measurable iff it is **constant on each atom of
\(\mathcal{F}\)**. We define \(X\) on \(\Omega=\{1,2,3,4\}\) and check both the
"constant on atoms" condition and the preimage condition for thresholds
\(\{X\leq x\}\).


In [ ]:
# A random variable X: Omega -> R, designed to be F-measurable.
# Atoms of F are {1,2} and {3,4}; X must be constant on each atom.
X = {1: 0.0, 2: 0.0, 3: 1.0, 4: 1.0}   # indicator of the block {3,4}

def preimage_le(x):
    '''{omega in Omega : X(omega) <= x} as a frozenset.'''
    return frozenset(w for w in Omega if X[w] <= x)

# (a) Measurable <=> constant on each atom of F.
atoms = [b for b in partition]
measurable_by_atoms = all(len({X[w] for w in atom}) == 1 for atom in atoms)
print("X constant on each atom of F?", measurable_by_atoms)

# (b) Preimage condition: {X <= x} in F for every threshold x.
thresholds = [-0.5, 0.0, 0.5, 1.0, 1.5]
preimages = {x: preimage_le(x) for x in thresholds}
print()
print("  x   {X<=x}      in F?")
for x in thresholds:
    pre = preimages[x]
    print(f" {x:>5}  {str(sorted(pre)):>14}   {pre in F}")

all_preimages_in_F = all(pre in F for pre in preimages.values())
print()
print("All preimages {X<=x} lie in F?", all_preimages_in_F)
assert measurable_by_atoms and all_preimages_in_F, "X must be F-measurable"
print("X is a valid random variable on (Omega, F).")


---
## Part 4 — Distribution of a random variable (pushforward measure)

The **distribution** of \(X\) is the probability measure \(P_X\) on
\((\mathbb{R},\mathcal{B})\) defined by

$$
P_X(B)\ =\ P\!\left(X^{-1}(B)\right)\ =\ P\!\left(\{\omega:X(\omega)\in B\}\right),
\qquad B\in\mathcal{B}.
$$

So \(P_X\) is the **pushforward** of \(P\) through \(X\). If \(Y=g(X)\) for
Borel-measurable \(g\), then \(P_Y\) is the pushforward of \(P_X\) through
\(g\) — the measure-theoretic statement of "applying a function to a random
variable transforms its distribution".

We compute \(P_X\) exactly on our finite space, then estimate it by Monte
Carlo sampling from \(P\) and compare empirical vs. theoretical masses.


In [ ]:
# Theoretical distribution of X: P_X({v}) = P({omega : X(omega) = v}).
values_X = sorted(set(X.values()))
P_X = {v: P(frozenset(w for w in Omega if X[w] == v)) for v in values_X}
print("Theoretical P_X:")
for v in values_X:
    print(f"  P_X({v}) = {P_X[v]:.2f}")

# Monte Carlo estimate of P_X by sampling omega ~ P.
rng = np.random.default_rng(42)
omega_list = sorted(Omega)
p_omega = np.array([masses[w] for w in omega_list])
N = 200_000
samples_omega = rng.choice(omega_list, size=N, p=p_omega)
X_arr = np.array([X[w] for w in samples_omega])
empirical = {v: float(np.mean(X_arr == v)) for v in values_X}

print()
print(f"{'value':>6} {'theory':>8} {'empirical':>10} {'abs err':>9}")
for v in values_X:
    err = abs(empirical[v] - P_X[v])
    print(f"{v:>6} {P_X[v]:>8.3f} {empirical[v]:>10.3f} {err:>9.4f}")

# Sanity check: total probability mass is 1.
assert abs(sum(P_X.values()) - 1.0) < 1e-12, "P_X must be a probability measure"
print()
print("Sum P_X =", sum(P_X.values()), "-> P_X is a probability measure on (R, B).")

# Pushforward through g: Y = g(X). Take g(x) = x^2 + 1 (Borel-measurable).
def g(x):
    return x**2 + 1

Y_arr = g(X_arr)
values_Y = sorted(set(g(v) for v in values_X))
P_Y_theory = {y: 0.0 for y in values_Y}
for v in values_X:
    P_Y_theory[g(v)] += P_X[v]
print()
print("Pushforward P_Y (Y = g(X) = X^2 + 1):")
for y in values_Y:
    emp = float(np.mean(Y_arr == y))
    print(f"  P_Y({y}) = {P_Y_theory[y]:.3f}   empirical = {emp:.3f}")
assert abs(sum(P_Y_theory.values()) - 1.0) < 1e-12
print("Pushforward preserved total mass = 1.")


---
## Part 5 — Preimages of Borel sets are events

Measurability guarantees every event defined through \(X\) lies in
\(\mathcal{F}\). For an interval \(B=[a,b]\subset\mathbb{R}\),

$$
X^{-1}(B)\ =\ \{\omega\in\Omega : a\leq X(\omega)\leq b\}\ \in\mathcal{F}.
$$

We verify that the preimage of an arbitrary Borel set (a union of intervals)
is one of the events in \(\mathcal{F}\), and we visualize the simulated
distribution of \(X\) alongside its theoretical pushforward \(P_X\).


In [ ]:
# Preimage of a Borel set B = [a, b] under X must be in F.
def preimage_interval(a, b):
    return frozenset(w for w in Omega if a <= X[w] <= b)

B_sets = [(-1.0, 0.5), (0.5, 1.5), (-0.5, 2.0), (0.0, 0.0), (1.0, 1.0)]
print("Borel set B      X^{-1}(B)          in F?")
for a, b in B_sets:
    pre = preimage_interval(a, b)
    print(f"  [{a:>4},{b:>4}]   {str(sorted(pre)):>14}      {pre in F}")

# A "union of intervals" Borel set: B = [-1,-0.5] U [0.5, 2]  (no mass here except 1.0)
B_union = frozenset(w for w in Omega if (X[w] <= -0.5) or (0.5 <= X[w] <= 2))
print()
print("B = [-1,-0.5] U [0.5,2]:  X^{-1}(B) =", sorted(B_union),
      " in F?", B_union in F)
assert B_union in F
print("Confirmed: preimage of a Borel set under a measurable X is an event in F.")

# Visualize: empirical histogram of X vs theoretical P_X.
fig, ax = plt.subplots(figsize=(8, 4.5))
vals = np.array(values_X)
theory_mass = np.array([P_X[v] for v in vals])
ax.bar(vals - 0.05, theory_mass, width=0.08, color="steelblue",
       label="theoretical $P_X$", alpha=0.7)
emp_mass = np.array([empirical[v] for v in vals])
ax.bar(vals + 0.05, emp_mass, width=0.08, color="crimson",
       label=f"empirical (N={N:,})", alpha=0.7)
ax.set_xlabel("value of X"); ax.set_ylabel("probability mass")
ax.set_title("Distribution of $X$ — pushforward of $P$")
ax.set_xticks(vals); ax.legend(); ax.set_ylim(0, 1.05)
plt.tight_layout(); plt.show()


**Your turn:** Define a finer sigma-algebra
\(\mathcal{F}'=2^{\Omega}\) (all subsets of \(\Omega\)). Construct a random
variable \(Z\) that is *not* measurable with respect to the original
\(\mathcal{F}\) (the partition-generated algebra) but *is* measurable with
respect to \(\mathcal{F}'\). Verify that some preimage \(\{Z\leq z\}\)
fails to lie in \(\mathcal{F}\) while it does lie in \(\mathcal{F}'\).


---
# Problem Set

**Try each problem before revealing the solution.**


## Problems

1. Let \(\Omega=\{a,b,c\}\) and let \(\mathcal{F}\) be the sigma-algebra
generated by the partition \(\{\{a\},\{b,c\}\}\). Enumerate every event in
\(\mathcal{F}\) and verify the three sigma-algebra axioms by hand.

2. On the probability space from Problem 1, assign masses
\(p_a=\tfrac12,\,p_b=\tfrac13,\,p_c=\tfrac16\). Define a random variable
\(X\) with \(X(a)=0,\,X(b)=X(c)=7\). Show \(X\) is \(\mathcal{F}\)-measurable
and compute its distribution \(P_X\) on \((\mathbb{R},\mathcal{B})\).

3. Let \(Y=g(X)\) with \(g(x)=x^2-2x\) and \(X\) as in Problem 2. Compute the
pushforward distribution \(P_Y\) analytically, then verify it by Monte Carlo
sampling from \(P\) (use a reproducible seed).

4. Give an example of a function \(Z:\Omega\to\mathbb{R}\) (with \(\Omega\)
and \(\mathcal{F}\) from Problem 1) that is **not** \(\mathcal{F}\)-measurable.
Identify a threshold \(z\) for which \(\{Z\leq z\}\notin\mathcal{F}\) and
explain in one sentence why this disqualifies \(Z\) as a random variable on
\((\Omega,\mathcal{F})\).

5. Explain in your own words why closure of \(\mathcal{F}\) under countable
unions (not just finite unions) is needed for the general theory even though on
a finite sample space finite unions suffice. Tie your answer to the
measurability condition \(\{X\leq x\}\in\mathcal{F}\).


<details>
<summary><strong>Reveal solutions</strong></summary>

**1.** The partition has 2 blocks, so \(\mathcal{F}=\sigma(\{a\},\{b,c\})\) has
\(2^{2}=4\) events — every union of blocks:

$$
\mathcal{F}=\Big\{\varnothing,\ \{a\},\ \{b,c\},\ \Omega\Big\}.
$$

Axioms: (i) \(\Omega\in\mathcal{F}\) ✓. (ii) Complements:
\(\{a\}^{c}=\{b,c\}\in\mathcal{F}\), \(\{b,c\}^{c}=\{a\}\in\mathcal{F}\),
\(\varnothing^{c}=\Omega\in\mathcal{F}\), \(\Omega^{c}=\varnothing\in\mathcal{F}\) ✓.
(iii) Finite unions of any two events in \(\mathcal{F}\) are again in
\(\mathcal{F}\) (check the 4×4 table; the only non-trivial union is
\(\{a\}\cup\{b,c\}=\Omega\in\mathcal{F}\)). On a finite space countable
unions reduce to a finite sub-union, so the axiom holds.

**2.** \(X\) is constant on each atom of \(\mathcal{F}\) (on \(\{a\}\) it equals 0;
on \(\{b,c\}\) it equals 7), so \(X\) is \(\mathcal{F}\)-measurable. Equivalently
\(\{X\leq x\}\in\mathcal{F}\) for every \(x\): for \(x<0\) it is
\(\varnothing\); for \(0\leq x<7\) it is \(\{a\}\); for \(x\geq 7\) it is
\(\Omega\). The distribution is

$$
P_X=\tfrac12\,\delta_{0}+\tfrac12\,\delta_{7},
\qquad P_X(\{0\})=\tfrac12,\ P_X(\{7\})=\tfrac12.
$$

**3.** \(g(0)=0^{2}-2\cdot0=0\) and \(g(7)=49-14=35\), so \(Y\) takes values
\(0\) (when \(X=0\)) and \(35\) (when \(X=7\)) with the same masses as \(X\):

$$
P_Y=\tfrac12\,\delta_{0}+\tfrac12\,\delta_{35}.
$$

Monte Carlo (seed 42): sample \(\omega\sim P\), apply \(X\) then \(g\), and
the empirical frequencies of 0 and 35 will concentrate on \(\tfrac12\) as
\(N\to\infty\). A reproducible check:

```python
rng = np.random.default_rng(42)
omega = np.array(["a","b","c"])
p = np.array([1/2, 1/3, 1/6])
s = rng.choice(omega, size=200_000, p=p)
Xv = np.where(s=="a", 0, 7)
Yv = Xv**2 - 2*Xv
print(np.mean(Yv==0), np.mean(Yv==35))  # ~0.5, ~0.5
```

**4.** Take \(Z(a)=0,\,Z(b)=1,\,Z(c)=2\). Then
\(\{Z\leq 1\}=\{a,b\}\). Is \(\{a,b\}\in\mathcal{F}\)? The events of
\(\mathcal{F}\) are \(\varnothing,\{a\},\{b,c\},\Omega\), so \(\{a,b\}\notin\mathcal{F}\).
Hence \(\{Z\leq 1\}\notin\mathcal{F}\) and \(Z\) is **not** measurable, i.e. not a
random variable on \((\Omega,\mathcal{F})\). It would become one if we enlarged
the sigma-algebra to \(2^{\Omega}\).

**5.** Real-valued random variables live on uncountable sample spaces (e.g.
\(\Omega=[0,1]\) with Lebesgue measure), where a single Borel set
\(B=\bigcup_{n}(a_n,b_n]\) is a *countable* union of intervals. For
\(\{X\in B\}=X^{-1}(B)=\bigcup_{n}\{a_n<X\leq b_n\}\) to be an event, each
\(\{a_n<X\leq b_n\}\in\mathcal{F}\) is not enough on its own — \(\mathcal{F}\)
must also be closed under the *countable* union that stitches them together.
On a finite \(\Omega\) every union is automatically finite, so the countable
condition is invisible; the general definition keeps the theory uniform across
discrete and continuous settings.


</details>
